# Logistic Regression — Chapter 4 (Math on Screen)

Chapter 4 wraps any plot animation in a **fixed template**: plot upper-left, math rows on the right, formula columns below.

## Abstractions (use for every clip)

| Module | Purpose |
|--------|---------|
| `tutorial_template.py` | `TutorialComposer`, `TutorialScene`, `TutorialTheme`, layout/typography/export |
| `handwrite_tutorial.py` | Patrick Hand + subscripts + symbol font + write-on reveal |
| `ch4_layout.py` | Chapter 4 defaults + `ch4_compose_tutorial_frame()` |

```python
scene = TutorialScene(plot=frame, math_right_blocks=..., math_bottom_blocks=...)
composer = make_composer("dark_rails")  # or any theme name
img = composer.render_scene(scene, write_progress=t)
```

All clips share **15.0×9.5 in @ 200 DPI → 3000×1900 px**. Typography and spacing live in `TutorialTypography`; colors/gradients in `TutorialTheme`.

## Exports

| Cell | Output |
|------|--------|
| **5** | Template PNG (`dark_rails`) |
| **6** | Handwrite demo + five theme MP4s |
| **7** | Likelihood story: `ch4_02` → `ch4_03` → `ch4_04` (3D path) → `ch4_05` (ball vectors) → `ch4_06` (GD steps) |


In [61]:
import importlib
import json
from pathlib import Path

import handwrite_tutorial
import tutorial_template
import ch4_layout

importlib.reload(handwrite_tutorial)
importlib.reload(tutorial_template)
importlib.reload(ch4_layout)
from ch4_layout import *



Chapter 4 layout OK — handwriting: Patrick Hand


In [62]:
# Reuse all Chapter 3 builders (datasets, knobs, triptych, strip, duo, …).
_CH3_NB = Path("logistic-regression-chap3.ipynb")
if not _CH3_NB.is_file():
    raise FileNotFoundError(_CH3_NB.resolve())
_ch3_src = "".join(json.loads(_CH3_NB.read_text())["cells"][1]["source"])
exec(compile(_ch3_src, str(_CH3_NB), "exec"), globals())
del _CH3_NB, _ch3_src



Chapter 3 setup OK — 20 clean, 26 with noise.


logistic-regression-chap3.ipynb:580: SyntaxWarning: invalid escape sequence '\s'
  "cell_type": "code",
logistic-regression-chap3.ipynb:584: SyntaxWarning: invalid escape sequence '\s'
  "outputs": [],


In [63]:
def ch4_sample_mistakes_triptych_frame():
    """One representative frame from the ch3_25 mistakes triptych (split-screen) family.

    Returns ``(plot_img, w_st, w_el, b)`` for math-rail values.
    """
    spec = CH3_LOSS_SPECS["mistakes"]
    loss_fn = spec["fn"]
    study, exam, y = study_sep, exam_sep, y_sep
    wr, we, br = ch3_triptych_script_weights()
    seqs, triples, losses = {}, {}, {}
    for which in ("st", "el", "b"):
        c = ch3_active_value(which, wr, we, br)
        seqs[which] = ch3_quad_sweep(c, CH3_LOSS_SYM_DELTA, CH3_SWEEP_NSEG)
        triples[which] = [ch3_triplet(which, float(v)) for v in seqs[which]]
        losses[which] = [loss_fn(ws, we, bb, study, exam, y) for ws, we, bb in triples[which]]
    all_l = np.concatenate([losses["st"], losses["el"], losses["b"]])
    pad_y = 0.06 * max(1e-6, float(np.nanmax(all_l) - np.nanmin(all_l)))
    y_lo = float(np.nanmin(all_l) - pad_y)
    y_hi = float(np.nanmax(all_l) + pad_y)

    def _xlim(seq):
        span = float(np.max(seq) - np.min(seq))
        pad_x = max(0.06 * span, 0.08)
        return float(np.min(seq) - pad_x), float(np.max(seq) + pad_x)

    ws, we, bb = triples["el"][len(triples["el"]) // 2]
    n_st = len(seqs["st"])
    n_el = len(seqs["el"])
    n_b = max(2, len(seqs["b"]) // 2)
    rots = _ch3_triptych_pack_knob_rots(ws, we, bb, wr, we, br)
    frame = ch3_triptych_frame(
        ws,
        we,
        bb,
        study,
        exam,
        y,
        spec,
        show_colormap=bool(spec["colormap"]),
        xs_st=seqs["st"][:n_st],
        ys_st=losses["st"][:n_st],
        xs_el=seqs["el"][:n_el],
        ys_el=losses["el"][:n_el],
        xs_b=seqs["b"][:n_b],
        ys_b=losses["b"][:n_b],
        x_lim_st=_xlim(seqs["st"]),
        x_lim_el=_xlim(seqs["el"]),
        x_lim_b=_xlim(seqs["b"]),
        y_lo=y_lo,
        y_hi=y_hi,
        knob_rots=rots,
        knob_scales=[1.0, float(CH3_KNOB_ACTIVE_SCALE), 1.0],
        arrows=None,
        panel_visible=(True, True, True),
        emphasize_knob="el",
    )
    return frame, float(ws), float(we), float(bb)


def ch4_math_right_blocks(w_st, w_el, b, study, exam, y):
    """Right-rail rows: current weights, gradient, and NLL at ``(w_st, w_el, b)``."""
    z = logits_plane(w_st, w_el, b, study, exam)
    p = sigmoid(z)
    yy = y.astype(float)
    eps = 1e-12
    nll = float(-np.sum(yy * np.log(p + eps) + (1.0 - yy) * np.log(1.0 - p + eps)))
    resid = p - yy
    gw1 = float(np.sum(resid * study))
    gw2 = float(np.sum(resid * exam))
    gb = float(np.sum(resid))
    w_text = rf"$w_1={w_st:.2f}$" + "\n" + rf"$w_2={w_el:.2f}$" + "\n" + rf"$b={b:.2f}$"
    g_text = (
        rf"$\partial w_1={gw1:.2f}$"
        + "\n"
        + rf"$\partial w_2={gw2:.2f}$"
        + "\n"
        + rf"$\partial b={gb:.2f}$"
    )
    return [
        {"label": r"$w,\,b$", "text": w_text, "bold_lhs": True, "role": "weights"},
        {"label": r"$\nabla\mathrm{NLL}$", "text": g_text, "bold_lhs": True, "role": "gradient"},
        {"label": "NLL", "text": rf"$NLL={nll:.2f}$", "bold_lhs": True, "role": "nll"},
    ]

CH4_BOTTOM_FORMULA_BLOCKS = [
    {"text": r"$p(y_i \mid x_i)=\hat p_i^{\,y_i}(1-\hat p_i)^{1-y_i}$", "bold_lhs": True, "role": "formula"},
    {"text": r"$\mathrm{NLL}(w)=-\sum_i \log p(y_i \mid x_i)$", "bold_lhs": True, "role": "formula"},
    {"text": r"$\nabla_w\,\mathrm{NLL}=\sum_i(\hat p_i-y_i)\,x_i$", "bold_lhs": True, "role": "formula"},
]


def ch4_tutorial_scene():
    """Shared plot + math blocks for static PNG and animated MP4."""
    plot, w1, w2, b = ch4_sample_mistakes_triptych_frame()
    return {
        "plot": plot,
        "math_right_blocks": ch4_math_right_blocks(w1, w2, b, study_sep, exam_sep, y_sep),
        "math_bottom_blocks": CH4_BOTTOM_FORMULA_BLOCKS,
    }


def ch4_render_tutorial_frame(scene, write_progress=1.0, *, theme=None, composer=None):
    return ch4_compose_tutorial_frame(
        scene["plot"],
        math_right_blocks=scene["math_right_blocks"],
        math_bottom_blocks=scene["math_bottom_blocks"],
        write_progress=write_progress,
        theme=theme,
        composer=composer,
    )


def ch4_export_handwrite_demo_mp4(n_frames=32, ms_per_frame=100):
    scene = ch4_tutorial_scene()
    return CH4_COMPOSER.export_mp4(
        ch4_scene_from_dict(scene),
        "ch4_01_handwrite_demo.mp4",
        save_mp4=save_mp4,
        output_dir=OUTPUT_DIR,
        n_frames=n_frames,
        ms_per_frame=ms_per_frame,
    )


def ch4_export_all_theme_demos(n_frames=32, ms_per_frame=100):
    """Export handwrite demo for each of the 5 color themes."""
    return ch4_export_theme_demos(
        ch4_tutorial_scene(),
        save_mp4=save_mp4,
        n_frames=n_frames,
        ms_per_frame=ms_per_frame,
    )

































# --- ch4_02: likelihood(w1,w2) landscape — duo left, tall 3D right ---
CH3_LIK_W12_W_ST0 = float(CH3_SCRIPT_K1_W_ST)
CH3_LIK_W12_W_EL0 = float(CH3_SCRIPT_K1_W_EL)
CH3_LIK_W12_B0 = float(CH3_SCRIPT_K1_B)
CH3_LIK_W12_W1_LO = float(CH3_SCRIPT_W1_WIDE[0])
CH3_LIK_W12_W1_HI = float(CH3_SCRIPT_W1_WIDE[1])
CH3_LIK_W12_W2_LO = float(CH3_LIK_W12_W_EL0 - CH3_SCRIPT_K1_STOP_SWEEP_DELTA)
CH3_LIK_W12_W2_HI = float(CH3_LIK_W12_W_EL0 + CH3_SCRIPT_K1_STOP_SWEEP_DELTA)
CH3_LIK_W12_B_HALF = float(CH3_SCRIPT_K1_TIGHT_SWEEP_DELTA * 20.0)
CH3_LIK_W12_FIGSIZE = EXPORT_FIGSIZE
CH3_LIK_W12_WIDTH_RATIOS = (1.22, 1.42)
CH3_LIK_W12_GRID_N = 36 if not _CH3_DRAFT else 22
CH3_LIK_W12_CURVE_LW = 2.4
CH3_LIK_W12_SURFACE_ALPHA = 0.88 * 0.9
CH3_LIK_W12_SLICE_ALPHA = 0.42 * 0.9
CH3_LIK_W12_MS = 90 if not _CH3_DRAFT else 110
CH3_LIK_W12_N_HOLD = max(10, CH3_SCRIPT_N_HOLD // 4)
CH3_LIK_W12_N_KNOB = max(28, _smooth_n(22))
CH3_LIK_W12_N_ROT = max(32, _smooth_n(24))
CH3_LIK_W12_N_REVEAL = max(40, _smooth_n(30))
CH3_LIK_W12_N_FILL_W1 = 10 if not _CH3_DRAFT else 6
CH3_LIK_W12_N_FILL_W2 = 10 if not _CH3_DRAFT else 6
CH3_LIK_W12_N_B_SLICES = 7 if not _CH3_DRAFT else 4
CH3_LIK_W12_N_ORBIT = max(48, _smooth_n(36))
CH3_LIK_W12_AZIM_W1 = 90.0
CH3_LIK_W12_AZIM_W2 = 180.0
CH3_LIK_W12_AZIM_BOTH = -54.0
CH3_LIK_W12_AZIM_STACK = 90.0
CH3_LIK_W12_ELEV_W1 = 0.0
CH3_LIK_W12_ELEV_W2 = 0.0
CH3_LIK_W12_ELEV_BOTH = 26.0
CH3_LIK_W12_ELEV_STACK = 0.0
CH3_LIK_W12_N_SQUISH = max(28, _smooth_n(22))
CH3_LIK_W12_CORNER_W1 = CH3_LIK_W12_W1_LO
CH3_LIK_W12_CORNER_W2 = CH3_LIK_W12_W2_LO


def ch3_lik_w12_mesh_pack(study, exam, y, b, *, w1_lo, w1_hi, w2_lo, w2_hi, grid_n=None):
    gn = int(CH3_LIK_W12_GRID_N if grid_n is None else grid_n)
    g1 = np.linspace(float(w1_lo), float(w1_hi), gn, dtype=np.float64)
    g2 = np.linspace(float(w2_lo), float(w2_hi), gn, dtype=np.float64)
    W1m, W2m = np.meshgrid(g1, g2, indexing="ij")
    bf = np.full(W1m.size, float(b), dtype=np.float64)
    Zf = _ch3_likelihood_on_flat_w12_grid(study, exam, y, W1m.ravel(), W2m.ravel(), bf)
    Z = Zf.reshape(W1m.shape)
    return {
        "W1m": W1m, "W2m": W2m, "Z": Z,
        "w1_lo": float(w1_lo), "w1_hi": float(w1_hi),
        "w2_lo": float(w2_lo), "w2_hi": float(w2_hi),
    }


def ch3_figure_lik_w12_3d():
    """Split screen: dataset+knobs (left), tall 3D likelihood ridge (right)."""
    fig = plt.figure(figsize=CH3_LIK_W12_FIGSIZE)
    gs = fig.add_gridspec(
        1, 2, width_ratios=CH3_LIK_W12_WIDTH_RATIOS, wspace=CH3_DUO_WSPACE,
    )
    g_left = GridSpecFromSubplotSpec(
        2, 1, subplot_spec=gs[0, 0], height_ratios=CH3_LEFT_HEIGHT_RATIOS, hspace=CH3_LEFT_HSPACE,
    )
    ax_data = fig.add_subplot(g_left[0,  0])
    g_k = GridSpecFromSubplotSpec(1, 3, subplot_spec=g_left[1, 0], wspace=CH3_KNOB_WSPACE)
    axes_k = tuple(fig.add_subplot(g_k[0, j]) for j in range(3))
    ax3d = fig.add_subplot(gs[0, 1], projection="3d")
    fig.subplots_adjust(left=0.05, right=0.97, top=0.93, bottom=0.06)
    _ch3_align_knob_axes_under_data(fig, ax_data, axes_k)
    ch3_layout_knob_axes_like_bridge_end(fig, ax_data, axes_k)
    return fig, ax_data, ax3d, axes_k


def ch3_lik_w12_z_limits(Z, *, scale=1.0):
    z_hi = float(np.nanmax(Z)) * float(scale)
    pad = 0.10 * max(z_hi, 1e-15)
    return 0.0, z_hi + pad


def ch3_lik_w12_facecolors_full(W1m, W2m, reveal_u, *, rgba):
    t = float(np.clip(float(reveal_u), 0.0, 1.0))
    fc = np.empty(W1m.shape + (4,), dtype=float)
    rgba = mpl.colors.to_rgba(rgba)
    fc[..., :] = rgba
    fc[..., 3] = rgba[3] * t
    return fc


def ch3_lik_w12_stack_z_lim(z_lik_hi, *, pad_frac=0.06):
    """Fixed display box: all stacked surfaces compress into [0, z_hi]."""
    z_hi = float(z_lik_hi)
    pad = float(pad_frac) * max(z_hi, 1.0)
    return 0.0, z_hi + pad


def ch3_lik_w12_squish_z(Z, layer_i, n_layers, z_lo, z_hi, z_ref):
    """Map mistake height into layer_i of n_layers equal slots in [z_lo, z_hi]."""
    n = max(float(n_layers), 1.0)
    span = float(z_hi) - float(z_lo)
    slot = span / n
    z_base = float(z_lo) + float(layer_i) * slot
    scale = slot / max(float(z_ref), 1e-9)
    return z_base + np.asarray(Z, dtype=float) * scale


def ch3_lik_w12_squish_scalar(z_val, layer_i, n_layers, z_lo, z_hi, z_ref):
    n = max(float(n_layers), 1.0)
    span = float(z_hi) - float(z_lo)
    slot = span / n
    z_base = float(z_lo) + float(layer_i) * slot
    scale = slot / max(float(z_ref), 1e-9)
    return z_base + float(z_val) * scale


def _ch3_lik_w12_z_at(W1m, W2m, Z, w1, w2):
    d = (np.asarray(W1m, dtype=float) - float(w1)) ** 2 + (np.asarray(W2m, dtype=float) - float(w2)) ** 2
    return float(np.ravel(np.asarray(Z, dtype=float))[int(np.nanargmin(d))])


def ch3_lik_w12_knob3_z_ticks(stack_layers, n_layers, z_lo, z_hi):
    """Tick positions at stacked-layer centers; labels are Knob 3 (b) values."""
    if not stack_layers:
        return None, None
    n = max(float(n_layers), 1.0)
    span = float(z_hi) - float(z_lo)
    slot = span / n
    layers = sorted(
        (
            sl for sl in stack_layers
            if float(sl.get("reveal", 1.0)) > 1e-4 and sl.get("b") is not None
        ),
        key=lambda sl: int(sl.get("layer_i", 0)),
    )
    if not layers:
        return None, None
    tick_z, tick_lbl = [], []
    for sl in layers:
        li = int(sl.get("layer_i", 0))
        tick_z.append(float(z_lo) + (float(li) + 0.5) * slot)
        tick_lbl.append(f"{float(sl['b']):.2g}")
    return tick_z, tick_lbl


def ch3_lik_w12_facecolors_diag(W1m, W2m, reveal_u, *, w1_lo, w1_hi, w2_lo, w2_hi, rgba):
    u1 = (W1m - float(w1_lo)) / max(float(w1_hi) - float(w1_lo), 1e-9)
    u2 = (W2m - float(w2_lo)) / max(float(w2_hi) - float(w2_lo), 1e-9)
    t = float(np.clip(float(reveal_u), 0.0, 1.0))
    mask = (u1 + u2) <= 2.0 * t + 1e-9
    fc = np.empty(W1m.shape + (4,), dtype=float)
    rgba = mpl.colors.to_rgba(rgba)
    fc[..., :] = rgba
    fc[..., 3] = rgba[3] * mask.astype(float)
    return fc


def ch3_frame_lik_w12_3d(
    study,
    exam,
    y,
    w_st,
    w_el,
    b,
    *,
    mesh_pack,
    z_lim,
    curves,
    elev,
    azim,
    emphasize_knob="st",
    landscape_reveal=0.0,
    landscape_rgba=None,
    b_slices=None,
    slice_reveal=0.0,
    show_curves=True,
    marker=True,
    marker_z_offset=0.0,
    stack_layers=None,
    stack_n_layers=1.0,
    z_lik_ref=None,
    show_axis_labels=True,
    flat_surface=None,
    z_label=None,
    knob_pack=None,
    knob_scales=None,
):
    spec = CH3_LOSS_SPECS["likelihood"]
    loss_fn = spec["fn"]
    w_st, w_el, b = float(w_st), float(w_el), float(b)
    W1m, W2m, Z = mesh_pack["W1m"], mesh_pack["W2m"], mesh_pack["Z"]
    w1_lo, w1_hi = mesh_pack["w1_lo"], mesh_pack["w1_hi"]
    w2_lo, w2_hi = mesh_pack["w2_lo"], mesh_pack["w2_hi"]
    z_lo_ax, z_hi_ax = float(z_lim[0]), float(z_lim[1])
    z_ref = float(np.nanmax(Z) if z_lik_ref is None else z_lik_ref)
    n_stack = max(float(stack_n_layers), 1.0)

    fig, ax_data, ax3d, axes_k = ch3_figure_lik_w12_3d()
    leg = legend_linear_equation_values_bold_param(w_st, w_el, b, emphasize_knob)
    ch3_draw_left_panel(
        ax_data, w_st, w_el, b, study, exam, y, leg,
        show_colormap=True, highlight_mistakes_flag=False,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    if knob_pack is None:
        knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    else:
        knob_rgbs, canvas_sides = knob_pack
    if knob_scales is None:
        scales = ch3_knob_scales_emphasize(emphasize_knob, CH3_KNOB_ACTIVE_SCALE)
    else:
        scales = list(knob_scales)
    ch3_draw_knob_row(
        fig, axes_k, w_st, w_el, b, emphasize_knob,
        knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(w_st, w_el, b),
        knob_scales=scales, ax_data=ax_data,
    )

    ax3d.cla()
    lr = landscape_rgba if landscape_rgba is not None else (FAIL_COLOR, CH3_LIK_W12_SURFACE_ALPHA)
    z_pt = None
    if flat_surface is not None:
        Wb = flat_surface["W1m"]
        W2b = flat_surface["W2m"]
        Zb = np.asarray(flat_surface["Z"], dtype=float)
        fc_b = ch3_lik_w12_facecolors_full(
            Wb, W2b, 1.0,
            rgba=(FAIL_COLOR, CH3_LIK_W12_SURFACE_ALPHA * float(flat_surface.get("alpha_scale", 1.0))),
        )
        ax3d.plot_surface(
            Wb, W2b, Zb, facecolors=fc_b, shade=False,
            linewidth=0, antialiased=False, rstride=1, cstride=1, zorder=1,
        )
        z_pt = float(
            flat_surface["marker_z"]
            if flat_surface.get("marker_z") is not None
            else _ch3_lik_w12_z_at(Wb, W2b, Zb, w_st, w_el)
        )
    elif float(landscape_reveal) > 1e-4:
        fc = ch3_lik_w12_facecolors_diag(
            W1m, W2m, landscape_reveal,
            w1_lo=w1_lo, w1_hi=w1_hi, w2_lo=w2_lo, w2_hi=w2_hi, rgba=lr,
        )
        ax3d.plot_surface(
            W1m, W2m, Z, facecolors=fc, shade=False,
            linewidth=0, antialiased=False, rstride=1, cstride=1, zorder=1,
        )
    if stack_layers:
        for sl in stack_layers:
            rev = float(sl.get("reveal", 1.0)) * float(slice_reveal)
            if rev < 1e-4:
                continue
            pack_b = sl["pack"]
            Wb, Zb = pack_b["W1m"], pack_b["Z"]
            li = int(sl.get("layer_i", 0))
            Zplot = ch3_lik_w12_squish_z(Zb, li, n_stack, z_lo_ax, z_hi_ax, z_ref)
            fc_b = ch3_lik_w12_facecolors_full(
                Wb, pack_b["W2m"], rev,
                rgba=(FAIL_COLOR, CH3_LIK_W12_SURFACE_ALPHA * float(sl.get("alpha_scale", 1.0))),
            )
            ax3d.plot_surface(
                Wb, pack_b["W2m"], Zplot, facecolors=fc_b, shade=False,
                linewidth=0, antialiased=False, rstride=1, cstride=1, zorder=int(sl.get("zorder", 3)),
            )
    elif b_slices:
        for sl in b_slices:
            bb = float(sl["b"])
            rev = float(sl.get("reveal", 1.0)) * float(slice_reveal)
            if rev < 1e-4:
                continue
            pack_b = sl.get("pack")
            if pack_b is None:
                pack_b = ch3_lik_w12_mesh_pack(
                    study, exam, y, bb, w1_lo=w1_lo, w1_hi=w1_hi, w2_lo=w2_lo, w2_hi=w2_hi,
                )
            Wb, Zb = pack_b["W1m"], pack_b["Z"]
            z0 = float(sl.get("z_base", 0.0))
            fc_b = ch3_lik_w12_facecolors_full(
                Wb, pack_b["W2m"], rev,
                rgba=(FAIL_COLOR, CH3_LIK_W12_SLICE_ALPHA * float(sl.get("alpha_scale", 1.0))),
            )
            ax3d.plot_surface(
                Wb, pack_b["W2m"], Zb + z0, facecolors=fc_b, shade=False,
                linewidth=0, antialiased=False, rstride=1, cstride=1, zorder=3,
            )
    if show_curves and curves:
        for cw1, cw2, cz in curves:
            cw1 = np.asarray(cw1, dtype=float)
            cw2 = np.asarray(cw2, dtype=float)
            cz = np.asarray(cz, dtype=float) + float(marker_z_offset)
            if cw1.size >= 2:
                ax3d.plot(
                    cw1, cw2, cz, color=FAIL_COLOR, linewidth=CH3_LIK_W12_CURVE_LW,
                    alpha=0.95, zorder=8,
                )
    if z_pt is None:
        z_raw = float(loss_fn(w_st, w_el, b, study, exam, y))
        if stack_layers:
            top_i = max(int(sl.get("layer_i", 0)) for sl in stack_layers if float(sl.get("reveal", 0)) > 1e-4)
            z_pt = ch3_lik_w12_squish_scalar(z_raw, top_i, n_stack, z_lo_ax, z_hi_ax, z_ref)
        else:
            z_pt = z_raw + float(marker_z_offset)
    if marker:
        ax3d.scatter(
            [w_st], [w_el], [z_pt],
            color=FAIL_COLOR, edgecolors="white", linewidths=2.0,
            s=260.0, depthshade=False, zorder=20,
        )
    ax3d.set_xlim(w1_lo, w1_hi)
    ax3d.set_ylim(w2_lo, w2_hi)
    ax3d.set_zlim(float(z_lim[0]), float(z_lim[1]))
    knob3_zticks, knob3_zlabels = ch3_lik_w12_knob3_z_ticks(stack_layers, n_stack, z_lo_ax, z_hi_ax)
    if show_axis_labels:
        ax3d.set_xlabel("Knob 1", fontsize=AXIS_LABEL_SIZE, labelpad=10)
        ax3d.set_ylabel("Knob 2", fontsize=AXIS_LABEL_SIZE, labelpad=10)
        if knob3_zticks:
            ax3d.set_zlabel("Knob 3", fontsize=AXIS_LABEL_SIZE, labelpad=10)
            ax3d.set_zticks(knob3_zticks)
            ax3d.set_zticklabels(knob3_zlabels)
        else:
            ax3d.set_zlabel(str(z_label or "Likelihood"), fontsize=AXIS_LABEL_SIZE, labelpad=10)
        ax3d.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
        ax3d.grid(True)
    else:
        ax3d.set_xlabel("")
        ax3d.set_ylabel("")
        ax3d.set_zlabel("")
        ax3d.set_xticklabels([])
        ax3d.set_yticklabels([])
        ax3d.set_zticklabels([])
    ax3d.view_init(elev=float(elev), azim=float(azim))
    return fig_to_image(fig, dpi=CH3_ANIM_DPI)


def ch3_lik_w12_frame_opening(mesh_pack, z_lim):
    """Opening frame (ch3_83 start / ch3_85 end): split screen, empty 3D, corner weights."""
    w1_c = float(CH3_LIK_W12_CORNER_W1)
    w2_c = float(CH3_LIK_W12_CORNER_W2)
    b0 = float(CH3_LIK_W12_B0)
    return ch3_frame_lik_w12_3d(
        study_sep, exam_sep, y_sep, w1_c, w2_c, b0,
        mesh_pack=mesh_pack, z_lim=z_lim, curves=[],
        elev=CH3_LIK_W12_ELEV_W1, azim=CH3_LIK_W12_AZIM_W1,
        emphasize_knob="st", show_curves=False, marker=False,
    )


def _ch3_lik_w12_trace_knob1(study, exam, y, w2_fix, b, w1_from, w1_to, n):
    w1s = np.linspace(float(w1_from), float(w1_to), int(n), dtype=float)
    z = [_ch3_likelihood_on_flat_w12_grid(study, exam, y, [w], [w2_fix], [b])[0] for w in w1s]
    return w1s, np.full_like(w1s, float(w2_fix)), np.asarray(z, dtype=float)


def _ch3_lik_w12_trace_knob2(study, exam, y, w1_fix, b, w2_from, w2_to, n):
    w2s = np.linspace(float(w2_from), float(w2_to), int(n), dtype=float)
    z = [_ch3_likelihood_on_flat_w12_grid(study, exam, y, [w1_fix], [w], [b])[0] for w in w2s]
    return np.full_like(w2s, float(w1_fix)), w2s, np.asarray(z, dtype=float)


def ch3_build_frames_likelihood_w12_landscape_story():
    study, exam, y = study_sep, exam_sep, y_sep
    w1_0 = float(CH3_LIK_W12_W_ST0)
    w2_0 = float(CH3_LIK_W12_W_EL0)
    b0 = float(CH3_LIK_W12_B0)
    w1_lo, w1_hi = float(CH3_LIK_W12_W1_LO), float(CH3_LIK_W12_W1_HI)
    w2_lo, w2_hi = float(CH3_LIK_W12_W2_LO), float(CH3_LIK_W12_W2_HI)
    w1_c = float(CH3_LIK_W12_CORNER_W1)
    w2_c = float(CH3_LIK_W12_CORNER_W2)

    mesh0 = ch3_lik_w12_mesh_pack(study, exam, y, b0, w1_lo=w1_lo, w1_hi=w1_hi, w2_lo=w2_lo, w2_hi=w2_hi)
    z_lik_hi = float(np.nanmax(mesh0["Z"]))
    z_lim_full = ch3_lik_w12_z_limits(mesh0["Z"], scale=1.0)
    n_trace = max(24, CH3_LIK_W12_N_KNOB)

    frames = []
    curves = []

    def emit(
        ws, we, bb, *, elev, azim, emp="st", lrev=0.0, show_curves=True,
        z_lim=None, marker=True, srev=0.0, marker_z_offset=0.0, stack_layers=None,
        stack_n_layers=1.0, z_lik_ref=None,
    ):
        fr = ch3_frame_lik_w12_3d(
            study, exam, y, ws, we, bb,
            mesh_pack=mesh0, z_lim=z_lim or z_lim_full,
            curves=list(curves) if show_curves else [],
            elev=elev, azim=azim, emphasize_knob=emp,
            landscape_reveal=lrev, slice_reveal=srev,
            show_curves=show_curves, marker=marker,
            marker_z_offset=marker_z_offset, stack_layers=stack_layers,
            stack_n_layers=stack_n_layers,
            z_lik_ref=z_lik_hi if z_lik_ref is None else z_lik_ref,
        )
        frames.append(fr)

    def hold(ws, we, bb, n, **kw):
        for _ in range(int(n)):
            emit(ws, we, bb, **kw)

    # 1–2: empty 3D at reveal corner (w1_lo, w2_lo)
    open_fr = ch3_lik_w12_frame_opening(mesh0, z_lim_full)
    for _ in range(CH3_LIK_W12_N_HOLD):
        frames.append(open_fr.copy())

    # 3: knob 1 — full w1 sweep along bottom edge (w2 = w2_lo)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_KNOB, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        w1c = ch3_lerp(w1_lo, w1_hi, u)
        cw1, cw2, cz = _ch3_lik_w12_trace_knob1(study, exam, y, w2_lo, b0, w1_lo, w1c, n_trace)
        if curves:
            curves[-1] = (cw1, cw2, cz)
        else:
            curves.append((cw1, cw2, cz))
        emit(w1c, w2_lo, b0, elev=CH3_LIK_W12_ELEV_W1, azim=CH3_LIK_W12_AZIM_W1, emp="st")
    w1_end, w2_edge = w1_hi, w2_lo
    hold(w1_end, w2_edge, b0, CH3_LIK_W12_N_HOLD // 2, elev=CH3_LIK_W12_ELEV_W1, azim=CH3_LIK_W12_AZIM_W1)
    curves[-1] = _ch3_lik_w12_trace_knob1(study, exam, y, w2_lo, b0, w1_lo, w1_hi, n_trace)

    # 3b: return knob 1 to start; marker walks back along the curve
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_KNOB, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        w1c = ch3_lerp(w1_hi, w1_lo, u)
        emit(w1c, w2_lo, b0, elev=CH3_LIK_W12_ELEV_W1, azim=CH3_LIK_W12_AZIM_W1, emp="st")
    hold(w1_c, w2_lo, b0, CH3_LIK_W12_N_HOLD // 2, elev=CH3_LIK_W12_ELEV_W1, azim=CH3_LIK_W12_AZIM_W1)

    # 4: rotate to knob-2 view (+90° from knob-1); marker stays at home corner
    az0, az1 = float(CH3_LIK_W12_AZIM_W1), float(CH3_LIK_W12_AZIM_W2)
    el0, el1 = float(CH3_LIK_W12_ELEV_W1), float(CH3_LIK_W12_ELEV_W2)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_ROT, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(
            w1_c, w2_lo, b0,
            elev=ch3_lerp(el0, el1, u), azim=ch3_lerp(az0, az1, u),
        )

    # 5: knob 2 — full w2 sweep along left edge (w1 = w1_lo), same corner
    curves.append(_ch3_lik_w12_trace_knob2(study, exam, y, w1_lo, b0, w2_lo, w2_lo, 2))
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_KNOB, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        w2c = ch3_lerp(w2_lo, w2_hi, u)
        cw1, cw2, cz = _ch3_lik_w12_trace_knob2(study, exam, y, w1_lo, b0, w2_lo, w2c, n_trace)
        curves[-1] = (cw1, cw2, cz)
        emit(w1_lo, w2c, b0, elev=CH3_LIK_W12_ELEV_W2, azim=CH3_LIK_W12_AZIM_W2, emp="el")
    hold(w1_lo, w2_hi, b0, CH3_LIK_W12_N_HOLD // 2, elev=CH3_LIK_W12_ELEV_W2, azim=CH3_LIK_W12_AZIM_W2)

    # 6: rotate to oblique view
    az_b, el_b = float(CH3_LIK_W12_AZIM_BOTH), float(CH3_LIK_W12_ELEV_BOTH)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_ROT, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(
            w1_lo, w2_hi, b0,
            elev=ch3_lerp(el1, el_b, u), azim=ch3_lerp(az1, az_b, u),
        )

    # 7: raster fill — w1 steps with full w2 sweeps, then w2 steps with full w1 sweeps
    w1_vals = np.linspace(w1_lo, w1_hi, int(CH3_LIK_W12_N_FILL_W1))
    w2_vals = np.linspace(w2_lo, w2_hi, int(CH3_LIK_W12_N_FILL_W2))
    n_seg = max(12, CH3_LIK_W12_N_KNOB // 2)
    for w1v in w1_vals:
        cw1, cw2, cz = _ch3_lik_w12_trace_knob2(study, exam, y, float(w1v), b0, w2_lo, w2_hi, n_seg)
        curves.append((cw1, cw2, cz))
        emit(float(w1v), w2_hi, b0, elev=el_b, azim=az_b, emp="st")
    for w2v in w2_vals:
        cw1, cw2, cz = _ch3_lik_w12_trace_knob1(study, exam, y, float(w2v), b0, w1_lo, w1_hi, n_seg)
        curves.append((cw1, cw2, cz))
        emit(w1_hi, float(w2v), b0, elev=el_b, azim=az_b, emp="el")

    hold(w1_hi, w2_hi, b0, CH3_LIK_W12_N_HOLD // 2, elev=el_b, azim=az_b)

    # 8: diagonal landscape reveal; fade curves out
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_REVEAL, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(
            w1_hi, w2_hi, b0, elev=el_b, azim=az_b,
            lrev=u, show_curves=(u < 0.82),
        )
    emit(w1_hi, w2_hi, b0, elev=el_b, azim=az_b, lrev=1.0, show_curves=False)

    # 9: lower camera to elev=0 (same azimuth as landscape), crossfade + squish 1→2
    el_s = float(CH3_LIK_W12_ELEV_STACK)
    az_land = float(CH3_LIK_W12_AZIM_BOTH)
    stack_layers = [{
        "pack": mesh0, "layer_i": 0, "reveal": 1.0, "alpha_scale": 1.0, "zorder": 1, "b": float(b0),
    }]
    b_vals = np.linspace(-CH3_LIK_W12_B_HALF, CH3_LIK_W12_B_HALF, int(CH3_LIK_W12_N_B_SLICES))
    b_packs = [
        ch3_lik_w12_mesh_pack(study, exam, y, float(bv), w1_lo=w1_lo, w1_hi=w1_hi, w2_lo=w2_lo, w2_hi=w2_hi)
        for bv in b_vals
    ]
    z_lik_hi = max(z_lik_hi, max(float(np.nanmax(p["Z"])) for p in b_packs))
    z_lim_stack = ch3_lik_w12_stack_z_lim(z_lik_hi)
    n_lower = int(CH3_LIK_W12_N_ROT)
    n_sq = int(CH3_LIK_W12_N_SQUISH)
    for tv in np.linspace(0.0, 1.0, n_lower + n_sq, endpoint=True):
        u_all = float(tv)
        u_el = min(1.0, u_all * (n_lower + n_sq) / max(n_lower, 1))
        u_sq = max(0.0, (u_all * (n_lower + n_sq) - n_lower) / max(n_sq, 1))
        u_el = ch3_knob_smoothstep(u_el)
        u_sq = ch3_knob_smoothstep(u_sq)
        n_eff = ch3_lerp(1.0, 2.0, u_sq)
        z_lo_t = ch3_lerp(z_lim_full[0], z_lim_stack[0], u_sq)
        z_hi_t = ch3_lerp(z_lim_full[1], z_lim_stack[1], u_sq)
        lrev = (1.0 - u_sq) if u_sq < 1.0 else 0.0
        emit(
            w1_hi, w2_hi, b0,
            elev=ch3_lerp(el_b, el_s, u_el), azim=az_land,
            lrev=lrev, show_curves=False,
            z_lim=(z_lo_t, z_hi_t),
            stack_layers=list(stack_layers) if u_sq > 1e-4 else None,
            stack_n_layers=n_eff, srev=1.0,
        )

    # 10: knob 3 — squish all layers to fit n+1, then reveal new surface on top
    n_rev = max(16, CH3_LIK_W12_N_REVEAL // 2)
    for bi, bv in enumerate(b_vals):
        n_before = float(1 + bi)
        n_after = float(2 + bi)
        for tv in np.linspace(0.0, 1.0, n_sq, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            n_eff = ch3_lerp(n_before, n_after, u)
            emit(
                w1_hi, w2_hi, float(b_vals[bi - 1]) if bi > 0 else b0,
                elev=el_s, azim=az_land, emp="b", lrev=0.0, show_curves=False,
                z_lim=z_lim_stack, stack_layers=list(stack_layers), srev=1.0,
                stack_n_layers=n_eff,
            )
        layer = {
            "pack": b_packs[bi], "layer_i": int(1 + bi), "reveal": 0.0,
            "alpha_scale": 1.0, "zorder": 4 + bi, "b": float(bv),
        }
        stack_layers.append(dict(layer))
        for tv in np.linspace(0.0, 1.0, n_rev, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            stack_layers[-1]["reveal"] = u
            emit(
                w1_hi, w2_hi, float(bv), elev=el_s, azim=az_land,
                emp="b", lrev=0.0, show_curves=False, z_lim=z_lim_stack,
                stack_layers=list(stack_layers), srev=1.0,
                stack_n_layers=n_after, marker=True,
            )
        stack_layers[-1]["reveal"] = 1.0
        hold(
            w1_hi, w2_hi, float(bv), CH3_LIK_W12_N_HOLD // 3,
            elev=el_s, azim=az_land, emp="b", lrev=0.0, show_curves=False,
            z_lim=z_lim_stack, stack_layers=list(stack_layers), srev=1.0,
            stack_n_layers=n_after,
        )

    # 11: orbit at elev=0
    az_start = az_land
    n_final = float(len(stack_layers))
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_ORBIT, endpoint=True):
        az = az_start + 360.0 * float(tv)
        emit(
            w1_hi, w2_hi, float(b_vals[-1]), elev=el_s, azim=az,
            emp="b", lrev=0.0, show_curves=False, z_lim=z_lim_stack,
            stack_layers=list(stack_layers), srev=1.0,             stack_n_layers=n_final,
        )
    if frames:
        last = frames[-1]
        for _ in range(CH3_LIK_W12_N_HOLD):
            frames.append(last.copy())
    return frames


def ch4_export_likelihood_w12_landscape():
    frames = ch3_build_frames_likelihood_w12_landscape_story()
    fn = "ch4_02_likelihood_w12_landscape.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_W12_MS))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


# --- ch4_03/04: likelihood ch4 notation/NLL morph → 3D measurements + trajectory ---

CH3_LIK_CH4_MS = 90 if not _CH3_DRAFT else 110
CH3_LIK_3D_MS = 90 if not _CH3_DRAFT else 110
CH3_LIK_CH4_N_MORPH = 8 if _CH3_DRAFT else max(28, _smooth_n(22))
CH3_LIK_CH4_N_LOG = 8 if _CH3_DRAFT else max(32, _smooth_n(24))
CH3_LIK_CH4_N_NLL = 8 if _CH3_DRAFT else max(32, _smooth_n(24))
CH3_LIK_CH4_N_NOTATION_MOVE = 8 if _CH3_DRAFT else max(24, _smooth_n(18))
CH3_LIK_CH4_N_KNOB_SWAP = 8 if _CH3_DRAFT else max(20, _smooth_n(14))
CH3_LIK_3D_N_TRANS = 10 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_3D_N_PATH = 24 if _CH3_DRAFT else max(120, _smooth_n(90))
CH3_LIK_3D_N_COLOR = 8 if _CH3_DRAFT else max(40, _smooth_n(32))
CH3_LIK_3D_CAM_AZIM0 = -128.0
CH3_LIK_3D_CAM_PATH_ROT = 90.0
CH3_LIK_3D_N_INTRO_ZOOM = 6 if _CH3_DRAFT else max(22, _smooth_n(16))
CH3_LIK_3D_N_INTRO_SPIN = 10 if _CH3_DRAFT else max(64, _smooth_n(48))
CH3_LIK_3D_N_INTRO_ZOOM_OUT = 6 if _CH3_DRAFT else max(22, _smooth_n(16))
CH3_LIK_3D_BALL_NSAMPLE = 16 if _CH3_DRAFT else 40
CH3_LIK_3D_BALL_R_SCALE = 0.085
CH3_LIK_3D_BALL_QUIVER_LEN = 0.075
CH3_LIK_3D_MS_BALL = 110 if not _CH3_DRAFT else 130
CH3_LIK_3D_AXIS_LABEL_SCALE = 1.45
CH3_LIK_3D_POINT_COLOR = FAIL_COLOR
CH3_LIK_3D_PATH_START = (-0.5, 0.33, -0.5)
CH3_LIK_GD_STEP = 0.06
CH3_LIK_GD_N_ITERS = 10 if not _CH3_DRAFT else 3
CH3_LIK_GD_N_HOLD_ARROWS = 4 if _CH3_DRAFT else 12
CH3_LIK_GD_N_PARAM_STEP = 4 if _CH3_DRAFT else 10
CH3_LIK_GD_SUBSTEPS = 6 if _CH3_DRAFT else 12
CH3_LIK_3D_MS_GD = 120 if not _CH3_DRAFT else 140

# ch4_02 end frame fills the canvas; morph shrinks it into the template plot slot.
CH4_LIK_PLOT_START_RECT = (0.0, 0.0, 1.0, 1.0)
CH3_LIK_CH4_SINGLE_ELEV = 18.0


def _ch3_lik_w12_z_limits_signed(Z, *, pad_frac=0.10):
    z_lo = float(np.nanmin(Z))
    z_hi = float(np.nanmax(Z))
    span = max(z_hi - z_lo, 1e-9)
    pad = float(pad_frac) * span
    return z_lo - pad, z_hi + pad


def _ch3_lik_w12_z_morph_limits(Zlik, Zlog, Znll, log_u, nll_u):
    """Blend z-axis limits: ℒ (0…max) → log ℒ (min…max) → NLL (0…max)."""
    mu_log = float(np.clip(log_u, 0.0, 1.0))
    mu_nll = float(np.clip(nll_u, 0.0, 1.0))
    lim_lik = ch3_lik_w12_z_limits(Zlik)
    lim_log = _ch3_lik_w12_z_limits_signed(Zlog)
    lim_nll = ch3_lik_w12_z_limits(Znll)
    if mu_nll > 1e-9:
        lim_a, lim_b, mu = lim_log, lim_nll, mu_nll
    elif mu_log > 1e-9:
        lim_a, lim_b, mu = lim_lik, lim_log, mu_log
    else:
        return lim_lik
    return (
        (1.0 - mu) * lim_a[0] + mu * lim_b[0],
        (1.0 - mu) * lim_a[1] + mu * lim_b[1],
    )


def _ch3_lik_w12_z_morph_surface(Zlik, Zlog, Znll, log_u, nll_u, z_lim):
    """Morph surface height in normalized z, with axis limits lerped separately."""
    mu_log = float(np.clip(log_u, 0.0, 1.0))
    mu_nll = float(np.clip(nll_u, 0.0, 1.0))
    lim_lik = ch3_lik_w12_z_limits(Zlik)
    lim_log = _ch3_lik_w12_z_limits_signed(Zlog)
    lim_nll = ch3_lik_w12_z_limits(Znll)
    zlo, zhi = float(z_lim[0]), float(z_lim[1])

    def _norm(Z, lo, hi):
        return (np.asarray(Z, dtype=float) - float(lo)) / max(float(hi) - float(lo), 1e-9)

    def _denorm(t):
        return t * (zhi - zlo) + zlo

    if mu_nll > 1e-9:
        t = (1.0 - mu_nll) * _norm(Zlog, *lim_log) + mu_nll * _norm(Znll, *lim_nll)
    elif mu_log > 1e-9:
        t = (1.0 - mu_log) * _norm(Zlik, *lim_lik) + mu_log * _norm(Zlog, *lim_log)
    else:
        t = _norm(Zlik, *lim_lik)
    return _denorm(t)


def _ch3_lik86_terminal_state():
    study, exam, y = study_sep, exam_sep, y_sep
    w1_lo = float(CH3_LIK_W12_W1_LO)
    w1_hi = float(CH3_LIK_W12_W1_HI)
    w2_lo = float(CH3_LIK_W12_W2_LO)
    w2_hi = float(CH3_LIK_W12_W2_HI)
    b0 = float(CH3_LIK_W12_B0)
    b_vals = np.linspace(-float(CH3_LIK_W12_B_HALF), float(CH3_LIK_W12_B_HALF), int(CH3_LIK_W12_N_B_SLICES))
    b_last = float(b_vals[-1])
    mesh = ch3_lik_w12_mesh_pack(study, exam, y, b_last, w1_lo=w1_lo, w1_hi=w1_hi, w2_lo=w2_lo, w2_hi=w2_hi)
    z_lim = ch3_lik_w12_z_limits(mesh["Z"], scale=1.0)
    return {
        "study": study, "exam": exam, "y": y,
        "w_st": w1_hi, "w_el": w2_hi, "b": b_last,
        "w1_lo": w1_lo, "w1_hi": w1_hi, "w2_lo": w2_lo, "w2_hi": w2_hi,
        "mesh": mesh, "z_lim": z_lim, "b_vals": b_vals,
    }


def _ch3_lik_w12_plot_crop(img):
    """Crop the right 3D panel from a split-screen ch3_86 frame."""
    img = img.convert("RGB")
    w, h = img.size
    x0 = int(round(w * 0.405))
    return img.crop((x0, int(h * 0.04), w - int(w * 0.02), h - int(h * 0.05)))


def ch3_frame_lik_w12_single_surface(
    state,
    *,
    log_u=0.0,
    nll_u=0.0,
    keep_layers=1,
    elev=None,
    azim=None,
    show_axis_labels=True,
    knob_labeled_blend=None,
):
    """One b-slice surface; morph Z: likelihood → log → NLL (negated log)."""
    study, exam, y = state["study"], state["exam"], state["y"]
    ws, we, bb = state["w_st"], state["w_el"], state["b"]
    mesh = state["mesh"]
    W1m, W2m, Z = mesh["W1m"], mesh["W2m"], mesh["Z"]
    Zlik = np.asarray(Z, dtype=float)
    Zlog = np.log(np.maximum(Zlik, 1e-18))
    Znll = -Zlog
    mu_log = float(np.clip(log_u, 0.0, 1.0))
    mu_nll = float(np.clip(nll_u, 0.0, 1.0))
    z_lim = _ch3_lik_w12_z_morph_limits(Zlik, Zlog, Znll, mu_log, mu_nll)
    Zmix = _ch3_lik_w12_z_morph_surface(Zlik, Zlog, Znll, mu_log, mu_nll, z_lim)
    marker_z = _ch3_lik_w12_z_at(W1m, W2m, Zmix, ws, we)
    if mu_nll > 0.5:
        z_lab = "NLL"
    elif mu_log > 0.5:
        z_lab = "log likelihood"
    else:
        z_lab = "Likelihood"
    el = float(CH3_LIK_CH4_SINGLE_ELEV if elev is None else elev)
    az = float(CH3_LIK_W12_AZIM_BOTH if azim is None else azim)
    knob_pack = None
    if knob_labeled_blend is not None:
        from ch4_layout import ch4_knob_asset_pack, ch4_knob_asset_pack_blended

        knob_pack = ch4_knob_asset_pack_blended(
            ch3_knob_asset_pack(),
            ch4_knob_asset_pack(),
            knob_labeled_blend,
        )
    return ch3_frame_lik_w12_3d(
        study, exam, y, ws, we, bb,
        mesh_pack=mesh, z_lim=z_lim, curves=[],
        elev=el, azim=az, emphasize_knob="all",
        landscape_reveal=0.0, show_curves=False, marker=True,
        stack_layers=None, stack_n_layers=1.0, slice_reveal=1.0,
        show_axis_labels=show_axis_labels,
        flat_surface={"W1m": W1m, "W2m": W2m, "Z": Zmix, "marker_z": marker_z},
        z_label=z_lab,
        knob_pack=knob_pack,
        knob_scales=[1.0, 1.0, 1.0],
    )


def ch3_build_frames_likelihood_ch4_nll_story():
    from ch4_layout import (
        CH4_COMPOSER,
        CH4_FORMULAS_SECTION_TITLE,
        CH4_NOTATION_SECTION_TITLE,
        ch4_blend_images,
        ch4_cached_notation_corner_blocks,
        ch4_formula_blocks_likelihood,
        ch4_formula_blocks_log_likelihood,
        ch4_formula_blocks_nll_story,
        ch4_group_write_from_slot,
        ch4_notation_blocks_basic,
        ch4_notation_blocks_expanded,
        compose_tutorial,
    )

    state = _ch3_lik86_terminal_state()
    frames = []
    style = CH4_COMPOSER.handwrite_style()

    def plot_surface(*, log_u=0.0, nll_u=0.0, knob_labeled_blend=None):
        return ch3_frame_lik_w12_single_surface(
            state,
            log_u=log_u,
            nll_u=nll_u,
            show_axis_labels=True,
            knob_labeled_blend=knob_labeled_blend,
        )

    def emit(
        plot_img,
        *,
        layout_u=1.0,
        panel_u=1.0,
        title_write_progress=None,
        write_progress=0.0,
        right_blocks=None,
        bottom_blocks=None,
        right_write_progress=None,
        bottom_write_progress=None,
        progress_override=None,
    ):
        frames.append(
            compose_tutorial(
                plot_img,
                right_blocks=right_blocks if right_blocks is not None else ch4_notation_blocks_basic(),
                bottom_blocks=bottom_blocks if bottom_blocks is not None else ch4_formula_blocks_likelihood(),
                right_title="Notation",
                bottom_title="Formulas",
                layout_u=layout_u,
                panel_u=panel_u,
                title_write_progress=title_write_progress,
                write_progress=write_progress,
                plot_start_rect=CH4_LIK_PLOT_START_RECT,
                right_write_progress=right_write_progress,
                bottom_write_progress=bottom_write_progress,
                progress_override=progress_override,
                theme="classic_light",
            )
        )

    plot0 = plot_surface()
    n_titles = max(14, _smooth_n(12))
    n_write = max(40, _smooth_n(32))
    n_expand = max(24, _smooth_n(18))
    basic_slots = 4  # 3 weight lines on right + 1 bottom formula line

    # 1 — full ch4_02 figure resizes into the template plot slot (no rails yet)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_MORPH, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(plot0, layout_u=u, panel_u=0.0, title_write_progress=0.0, write_progress=0.0)

    # 2 — section titles appear and stay (rails visible, no block text yet)
    for tv in np.linspace(0.0, 1.0, n_titles, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(plot0, layout_u=1.0, panel_u=1.0, title_write_progress=u, write_progress=0.0)

    # 3 — handwrite basic notation + likelihood formula
    for tv in np.linspace(0.0, 1.0, n_write, endpoint=True):
        emit(
            plot0,
            layout_u=1.0,
            panel_u=1.0,
            title_write_progress=1.0,
            write_progress=ch3_knob_smoothstep(float(tv)),
        )

    # 4 — add expanded notation lines (y_i, x_i,ST, …); keep likelihood formula
    right_exp = ch4_notation_blocks_expanded()
    bottom_lik = ch4_formula_blocks_likelihood()
    for tv in np.linspace(0.0, 1.0, n_expand, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        prog = ch4_group_write_from_slot([right_exp, bottom_lik], basic_slots, u, style=style)
        emit(
            plot0,
            layout_u=1.0,
            panel_u=1.0,
            title_write_progress=1.0,
            write_progress=1.0,
            right_blocks=right_exp,
            bottom_blocks=bottom_lik,
            progress_override={"right": prog[0], "bottom": prog[1]},
        )

    # 5 — morph surface ℒ → log ℒ; rewrite bottom formula only
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_LOG, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(
            plot_surface(log_u=u, nll_u=0.0),
            layout_u=1.0,
            panel_u=1.0,
            title_write_progress=1.0,
            write_progress=1.0,
            right_blocks=right_exp,
            bottom_blocks=ch4_formula_blocks_log_likelihood(),
            right_write_progress=1.0,
            bottom_write_progress=u,
        )

    # 6 — morph log ℒ → NLL; rewrite bottom formula only
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_NLL, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(
            plot_surface(log_u=1.0, nll_u=u),
            layout_u=1.0,
            panel_u=1.0,
            title_write_progress=1.0,
            write_progress=1.0,
            right_blocks=right_exp,
            bottom_blocks=ch4_formula_blocks_nll_story(),
            right_write_progress=1.0,
            bottom_write_progress=u,
        )

    # 7 — move notation from right rail to corner (ch4_04 layout); knobs still numbered
    plot_nll = plot_surface(log_u=1.0, nll_u=1.0)
    bottom_nll = ch4_formula_blocks_nll_story()
    corner = ch4_cached_notation_corner_blocks()
    frame_right_notation = compose_tutorial(
        plot_nll,
        right_blocks=right_exp,
        bottom_blocks=bottom_nll,
        right_title="Notation",
        bottom_title="Formulas",
        layout_u=1.0,
        panel_u=1.0,
        write_progress=1.0,
        plot_start_rect=CH4_LIK_PLOT_START_RECT,
        theme="classic_light",
    )
    frame_corner_notation = compose_tutorial(
        plot_nll,
        right_blocks=[],
        bottom_blocks=bottom_nll,
        corner_blocks=corner,
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        layout_u=1.0,
        panel_u=1.0,
        write_progress=1.0,
        plot_start_rect=CH4_LIK_PLOT_START_RECT,
        theme="classic_light",
    )
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_NOTATION_MOVE, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(ch4_blend_images(frame_right_notation, frame_corner_notation, u))

    # 8–10 — swap numbered knobs → w_ST / w_EL / b one at a time
    def emit_corner_nll(plot_img):
        frames.append(
            compose_tutorial(
                plot_img,
                right_blocks=[],
                bottom_blocks=bottom_nll,
                corner_blocks=corner,
                bottom_title=CH4_FORMULAS_SECTION_TITLE,
                corner_title=CH4_NOTATION_SECTION_TITLE,
                layout_u=1.0,
                panel_u=1.0,
                write_progress=1.0,
                plot_start_rect=CH4_LIK_PLOT_START_RECT,
                theme="classic_light",
            )
        )

    for slot in range(3):
        for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_KNOB_SWAP, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            blends = tuple(1.0 if i < slot else (u if i == slot else 0.0) for i in range(3))
            emit_corner_nll(plot_surface(log_u=1.0, nll_u=1.0, knob_labeled_blend=blends))

    if frames:
        last = frames[-1]
        for _ in range(max(8, CH3_SCRIPT_N_HOLD // 4)):
            frames.append(last.copy())
    return frames


def _ch3_lik_fun_path_3d(n_pts=None, start=None):
    """Playful looping path in (w_ST, w_EL, b) weight space."""
    n_pts = int(CH3_LIK_3D_N_PATH if n_pts is None else n_pts)
    t = np.linspace(0.0, 1.0, n_pts, endpoint=True)
    if start is None:
        start = CH3_LIK_3D_PATH_START
    w1_c, w2_c, b_c = (float(start[0]), float(start[1]), float(start[2]))
    w1 = w1_c + 1.8 * np.sin(2.0 * np.pi * t) * (0.35 + 0.65 * t)
    w2 = w2_c + 1.6 * np.cos(2.4 * np.pi * t + 0.6) * (0.35 + 0.65 * t)
    b = b_c + 0.55 * np.sin(4.0 * np.pi * t + 1.1)
    return np.column_stack([w1, w2, b]).astype(float)


def _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib):
    """3-D axis labels (+45%) and sparser z ticks."""
    from matplotlib.ticker import MaxNLocator

    fs = float(AXIS_LABEL_SIZE) * float(CH3_LIK_3D_AXIS_LABEL_SCALE)
    ax3d.set_xlim(dlo1, dhi1)
    ax3d.set_ylim(dlo2, dhi2)
    ax3d.set_zlim(dlob, dhib)
    ax3d.set_xlabel(r"$w_{\mathrm{ST}}$", fontsize=fs, labelpad=8)
    ax3d.set_ylabel(r"$w_{\mathrm{EL}}$", fontsize=fs, labelpad=8)
    ax3d.set_zlabel(r"$b$", fontsize=fs, labelpad=8)
    ax3d.zaxis.set_major_locator(MaxNLocator(nbins=5))


def _ch3_lik_ax3d_fixed_bounds(ref_lo1, hi1, lo2, hi2, lob, hib, path_xyz, *, extra_pts=None):
    """Axis limits that contain the full path (and grid), with padding."""
    ref_lo = np.array([float(ref_lo1), float(lo2), float(lob)], dtype=float)
    ref_hi = np.array([float(hi1), float(hi2), float(hib)], dtype=float)
    ref_span = np.maximum(ref_hi - ref_lo, 1e-9)
    pts = [np.asarray(path_xyz, dtype=float).reshape(-1, 3)]
    if extra_pts is not None:
        pts.append(np.asarray(extra_pts, dtype=float).reshape(-1, 3))
    P = np.vstack(pts)
    P = P[np.isfinite(P).all(axis=1)]
    if P.shape[0] == 0:
        return float(ref_lo1), float(hi1), float(lo2), float(hi2), float(lob), float(hib)
    lo = np.minimum(P.min(axis=0), ref_lo)
    hi = np.maximum(P.max(axis=0), ref_hi)
    span = np.maximum(hi - lo, CH3_KERAS_NLL3D_ZOOM_MIN_SPAN_FRAC * ref_span)
    center = 0.5 * (lo + hi)
    lo = center - 0.5 * span
    hi = center + 0.5 * span
    pad = CH3_KERAS_NLL3D_VIEW_PAD_FRAC * (hi - lo)
    return (
        float(lo[0] - pad[0]),
        float(hi[0] + pad[0]),
        float(lo[1] - pad[1]),
        float(hi[1] + pad[1]),
        float(lo[2] - pad[2]),
        float(hi[2] + pad[2]),
    )


def _ch3_lik_cam_azim(cam_u, *, total_deg=CH3_LIK_3D_CAM_PATH_ROT, base=CH3_LIK_3D_CAM_AZIM0):
    u = float(np.clip(float(cam_u), 0.0, 1.0))
    return float(base + float(total_deg) * u)


def _ch3_lik_lerp_bounds(bounds_a, bounds_b, u):
    u = float(np.clip(float(u), 0.0, 1.0))
    a = tuple(float(v) for v in bounds_a)
    b = tuple(float(v) for v in bounds_b)
    return tuple(float(x + (y - x) * u) for x, y in zip(a, b))


def _ch3_lik_bounds_span(bounds):
    return np.array(
        [bounds[1] - bounds[0], bounds[3] - bounds[2], bounds[5] - bounds[4]],
        dtype=float,
    )


def _ch3_lik_ball_zoom_bounds(ws, we, bb, wide_bounds, *, r_scale=CH3_LIK_3D_BALL_R_SCALE):
    """Tight axis limits framing the NLL ball around one weight-space point."""
    center = np.array([float(ws), float(we), float(bb)], dtype=float)
    span_ref = float(np.max(_ch3_lik_bounds_span(wide_bounds)))
    R = float(r_scale) * span_ref
    half = max(R * 2.35, span_ref * 0.055)
    lo = center - half
    hi = center + half
    pad = 0.08 * (hi - lo)
    return (
        float(lo[0] - pad[0]), float(hi[0] + pad[0]),
        float(lo[1] - pad[1]), float(hi[1] + pad[1]),
        float(lo[2] - pad[2]), float(hi[2] + pad[2]),
    )


def _ch3_lik_ball_vector_field(study, exam, y, ws, we, bb, wide_bounds):
    """Sample points on a sphere + negative-NLL gradient vectors at each sample."""
    span_ref = float(np.max(_ch3_lik_bounds_span(wide_bounds)))
    R = float(CH3_LIK_3D_BALL_R_SCALE) * span_ref
    offs = _ch3_ball_unit_offsets(int(CH3_LIK_3D_BALL_NSAMPLE))
    center = np.array([float(ws), float(we), float(bb)], dtype=float)
    P = center[np.newaxis, :] + offs * R
    L = _ch3_nll_sum_on_flat_grid(study, exam, y, P[:, 0], P[:, 1], P[:, 2])
    U = np.empty(P.shape[0], dtype=float)
    V = np.empty(P.shape[0], dtype=float)
    W = np.empty(P.shape[0], dtype=float)
    for i, p in enumerate(P):
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, p[0], p[1], p[2])
        U[i], V[i], W[i] = -float(g1), -float(g2), -float(gb)
    return P, L, U, V, W


_BALL_FIELD_CACHE: dict[tuple, tuple] = {}


def _ch3_lik_ball_vector_field_cached(study, exam, y, ws, we, bb, wide_bounds):
    key = (
        round(float(ws), 5), round(float(we), 5), round(float(bb), 5),
        round(float(wide_bounds[0]), 4), round(float(wide_bounds[1]), 4),
        round(float(wide_bounds[2]), 4), round(float(wide_bounds[3]), 4),
        round(float(wide_bounds[4]), 4), round(float(wide_bounds[5]), 4),
        int(CH3_LIK_3D_BALL_NSAMPLE),
    )
    hit = _BALL_FIELD_CACHE.get(key)
    if hit is not None:
        return hit
    out = _ch3_lik_ball_vector_field(study, exam, y, ws, we, bb, wide_bounds)
    _BALL_FIELD_CACHE[key] = out
    return out


def _ch3_lik_ball_nll_limits(study, exam, y, path, wide_bounds):
    chunks = []
    for r in np.asarray(path, dtype=float):
        _, L, _, _, _ = _ch3_lik_ball_vector_field_cached(
            study, exam, y, float(r[0]), float(r[1]), float(r[2]), wide_bounds,
        )
        chunks.append(L)
    flat = np.concatenate(chunks) if chunks else np.array([0.0])
    return float(np.min(flat)), float(np.max(flat))


def _ch3_lik_draw_gd_step_arrows(ax3d, ws, we, bb, grad, eta, *, visible=(True, True, True)):
    """Axis-aligned GD step arrows: length = α × ∂NLL/∂(coord), direction of the update."""
    from ch4_layout import CH4_GD_ARROW_B_COLOR, CH4_GD_ARROW_EL_COLOR, CH4_GD_ARROW_ST_COLOR

    g1, g2, gb = (float(grad[0]), float(grad[1]), float(grad[2]))
    eta = float(eta)
    specs = (
        (-eta * g1, 0.0, 0.0, CH4_GD_ARROW_ST_COLOR, bool(visible[0])),
        (0.0, -eta * g2, 0.0, CH4_GD_ARROW_EL_COLOR, bool(visible[1])),
        (0.0, 0.0, -eta * gb, CH4_GD_ARROW_B_COLOR, bool(visible[2])),
    )
    for du, dv, dw, color, show in specs:
        if not show:
            continue
        mag = float(np.hypot(du, np.hypot(dv, dw)))
        if mag < 1e-12:
            continue
        ax3d.quiver(
            float(ws), float(we), float(bb), du, dv, dw,
            color=color, arrow_length_ratio=0.22, linewidth=2.8,
            normalize=False, alpha=0.95,
        )


def _ch3_lik_gd_frame_specs(pack):
    """Frame descriptors for sequential per-parameter GD (10 iterations)."""
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws, we, bb = (float(pack["gd_start"][0]), float(pack["gd_start"][1]), float(pack["gd_start"][2]))
    eta = float(pack["gd_eta"])
    n_iters = int(pack["gd_n_iters"])
    hold_n = max(int(CH3_LIK_GD_N_HOLD_ARROWS), 1)
    step_n = max(int(CH3_LIK_GD_N_PARAM_STEP), 2)
    specs = []
    trail = [(ws, we, bb)]

    def _append(ws_i, we_i, bb_i, grad, *, arrows, bold, trail_pts):
        specs.append({
            "ws": float(ws_i), "we": float(we_i), "bb": float(bb_i),
            "grad": (float(grad[0]), float(grad[1]), float(grad[2])),
            "eta": eta,
            "arrows": tuple(bool(v) for v in arrows),
            "bold": bold,
            "trail": list(trail_pts),
        })

    for _ in range(n_iters):
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        grad = (g1, g2, gb)
        for _ in range(hold_n):
            _append(ws, we, bb, grad, arrows=(True, True, True), bold=None, trail_pts=trail)

        ws0 = ws
        for si in range(step_n):
            u = ch3_knob_smoothstep(float(si) / float(step_n - 1))
            _append(
                ws0 - u * eta * g1, we, bb, grad,
                arrows=(False, True, True), bold=0, trail_pts=trail,
            )
        ws = ws0 - eta * g1

        we0 = we
        for si in range(step_n):
            u = ch3_knob_smoothstep(float(si) / float(step_n - 1))
            _append(
                ws, we0 - u * eta * g2, bb, grad,
                arrows=(False, False, True), bold=1, trail_pts=trail,
            )
        we = we0 - eta * g2

        bb0 = bb
        for si in range(step_n):
            u = ch3_knob_smoothstep(float(si) / float(step_n - 1))
            _append(
                ws, we, bb0 - u * eta * gb, grad,
                arrows=(False, False, False), bold=2, trail_pts=trail,
            )
        bb = bb0 - eta * gb
        trail.append((ws, we, bb))

    return specs


def _ch3_lik_draw_ball_vectors(
    ax3d, study, exam, y, ws, we, bb, wide_bounds, *,
    vmin, vmax, cmap=None, alpha=0.86,
    ball_field=None,
):
    if ball_field is None:
        ball_field = _ch3_lik_ball_vector_field_cached(study, exam, y, ws, we, bb, wide_bounds)
    P, L, U, V, W = ball_field
    gn = np.sqrt(U * U + V * V + W * W)
    keep = gn > 1e-14
    if not np.any(keep):
        return
    P = P[keep]
    L = L[keep]
    U = U[keep]
    V = V[keep]
    W = W[keep]
    cmap = mpl.colormaps.get_cmap("viridis") if cmap is None else cmap
    span = max(float(vmax) - float(vmin), 1e-9)
    cols = cmap((L - float(vmin)) / span)
    ax3d.scatter(
        P[:, 0], P[:, 1], P[:, 2],
        c=L, cmap=cmap, vmin=float(vmin), vmax=float(vmax),
        s=22.0, alpha=float(alpha) * 0.55, linewidths=0, depthshade=False, zorder=6,
    )
    ax3d.quiver(
        P[:, 0], P[:, 1], P[:, 2],
        U, V, W,
        colors=cols,
        length=float(CH3_LIK_3D_BALL_QUIVER_LEN),
        normalize=True,
        alpha=float(alpha),
        linewidth=0.95,
        arrow_length_ratio=0.34,
        zorder=7,
    )


def _ch3_lik_3d_measurements_pack(*, ball_colormap_limits=False):
    state = _ch3_lik86_terminal_state()
    study, exam, y = state["study"], state["exam"], state["y"]
    ws, we, bb = state["w_st"], state["w_el"], state["b"]
    gn = 10 if _CH3_DRAFT else 14
    w1g = np.linspace(float(state["w1_lo"]), float(state["w1_hi"]), gn)
    w2g = np.linspace(float(state["w2_lo"]), float(state["w2_hi"]), gn)
    bg = np.linspace(-float(CH3_LIK_W12_B_HALF), float(CH3_LIK_W12_B_HALF), gn)
    W1m, W2m, Bm = np.meshgrid(w1g, w2g, bg, indexing="ij")
    Lf = _ch3_nll_sum_on_flat_grid(study, exam, y, W1m.ravel(), W2m.ravel(), Bm.ravel()).reshape(W1m.shape)
    vmin, vmax = float(np.nanmin(Lf)), float(np.nanmax(Lf))
    k_lo1, k_hi1 = float(W1m.min()), float(W1m.max())
    k_lo2, k_hi2 = float(W2m.min()), float(W2m.max())
    k_lob, k_hib = float(Bm.min()), float(Bm.max())
    path = _ch3_lik_fun_path_3d(start=CH3_LIK_3D_PATH_START)
    ax3d_bounds = _ch3_lik_ax3d_fixed_bounds(
        k_lo1, k_hi1, k_lo2, k_hi2, k_lob, k_hib, path,
        extra_pts=np.array([[path[0, 0], path[0, 1], path[0, 2]]], dtype=float),
    )
    path_nll = np.array([
        float(-loss_log_likelihood(float(r[0]), float(r[1]), float(r[2]), study, exam, y))
        for r in path
    ], dtype=float)
    if ball_colormap_limits:
        ball_vmin, ball_vmax = _ch3_lik_ball_nll_limits(study, exam, y, path, ax3d_bounds)
    else:
        ball_vmin, ball_vmax = float(np.min(path_nll)), float(np.max(path_nll))
    span = max(float(ball_vmax) - float(ball_vmin), 1e-9)
    vir = mpl.colormaps.get_cmap("viridis")
    path_colors = [vir(float((v - ball_vmin) / span)) for v in path_nll]
    n_slow = max(CH3_LIK_3D_N_PATH // 3, 20)
    n_fast = CH3_LIK_3D_N_PATH - n_slow
    path_us = list(np.linspace(0.02, 0.35, n_slow, endpoint=True)) + list(
        np.linspace(0.35, 1.0, n_fast, endpoint=True)
    )
    return {
        "study": study, "exam": exam, "y": y,
        "W1m": W1m, "W2m": W2m, "Bm": Bm, "Lf": Lf,
        "vmin": vmin, "vmax": vmax,
        "path": path, "path_colors": path_colors,
        "ax3d_bounds": ax3d_bounds,
        "path_us": path_us, "n_slow": n_slow, "n_fast": n_fast,
        "ball_vmin": ball_vmin, "ball_vmax": ball_vmax,
    }


def ch3_frame_lik_weight3d_measurements(
    study, exam, y, ws, we, bb, *,
    W1m, W2m, Bm, Lf, vmin, vmax,
    path_xyz=None,
    path_colors=None,
    path_u=1.0,
    notation_condensed=False,
    measurements=None,
    write_progress=1.0,
    plot_alpha=1.0,
    ax3d_bounds=None,
    show_weight_grid=False,
    show_ball_vectors=False,
    wide_bounds=None,
    ball_vmin=None,
    ball_vmax=None,
    ball_field=None,
    elev=24.0,
    azim=None,
    cam_azim_u=0.0,
    cam_rot_deg=CH3_LIK_3D_CAM_PATH_ROT,
    grad=None,
    step_size=None,
    gd_formulas=False,
    gd_arrows_grad=None,
    gd_arrows_visible=None,
    gd_bold_update_idx=None,
    gd_bottom_blocks=None,
):
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_HERE_SECTION_TITLE,
        CH4_NOTATION_SECTION_TITLE,
        ch4_cached_formula_blocks_3d_story,
        ch4_cached_formula_blocks_gd_story,
        ch4_cached_notation_corner_blocks,
        ch4_formula_blocks_gd_story,
        ch4_knob_asset_pack,
        ch4_rails_cache_key,
        ch4_we_are_here_blocks,
        compose_tutorial,
    )

    fig, ax_data, ax3d, axes_k = ch3_figure_nllkeras_dataset_weight3d()
    leg = legend_linear_equation_values_bold_param(ws, we, bb, "all")
    ch3_draw_left_panel(ax_data, ws, we, bb, study, exam, y, leg, show_colormap=True, highlight_mistakes_flag=False)
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    knob_rgbs, canvas_sides = ch4_knob_asset_pack()
    ch3_draw_knob_row(
        fig, axes_k, ws, we, bb, "st", knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(ws, we, bb), knob_scales=[1.0, 1.0, 1.0], ax_data=ax_data,
    )
    if show_weight_grid:
        ax3d.scatter(
            W1m.ravel(), W2m.ravel(), Bm.ravel(),
            c=Lf.ravel(), cmap="viridis", vmin=float(vmin), vmax=float(vmax),
            s=28.0, alpha=0.45, linewidths=0, depthshade=False, zorder=1,
        )
    k_lo1, k_hi1 = float(W1m.min()), float(W1m.max())
    k_lo2, k_hi2 = float(W2m.min()), float(W2m.max())
    k_lob, k_hib = float(Bm.min()), float(Bm.max())
    if ax3d_bounds is not None:
        dlo1, dhi1, dlo2, dhi2, dlob, dhib = ax3d_bounds
    else:
        dlo1, dhi1, dlo2, dhi2, dlob, dhib = _ch3_keras_ax3d_zoom_bounds(
            k_lo1, k_hi1, k_lo2, k_hi2, k_lob, k_hib, np.array([[ws, we, bb]], dtype=float),
        )
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    if azim is None:
        azim = _ch3_lik_cam_azim(cam_azim_u, total_deg=float(cam_rot_deg))
    ax3d.view_init(elev=float(elev), azim=float(azim))
    ref_bounds = wide_bounds if wide_bounds is not None else (
        ax3d_bounds if ax3d_bounds is not None else (dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    )
    if gd_arrows_grad is not None:
        vis = gd_arrows_visible if gd_arrows_visible is not None else (True, True, True)
        _ch3_lik_draw_gd_step_arrows(
            ax3d, ws, we, bb, gd_arrows_grad, float(step_size if step_size is not None else CH3_LIK_GD_STEP),
            visible=vis,
        )
    elif show_ball_vectors:
        _ch3_lik_draw_ball_vectors(
            ax3d, study, exam, y, ws, we, bb, ref_bounds,
            vmin=float(ball_vmin if ball_vmin is not None else vmin),
            vmax=float(ball_vmax if ball_vmax is not None else vmax),
            ball_field=ball_field,
        )
    ax3d.scatter(
        [ws], [we], [bb], s=320, c=[CH3_LIK_3D_POINT_COLOR], edgecolors="white",
        linewidths=2.0, depthshade=False, zorder=20,
    )
    if path_xyz is not None and len(path_xyz) >= 2:
        P = np.asarray(path_xyz, dtype=float)
        n_keep = max(2, int(round(float(path_u) * (P.shape[0] - 1))) + 1)
        Pk = P[:n_keep]
        if path_colors is not None and len(path_colors) >= n_keep:
            cols = np.asarray(path_colors[:n_keep])
            for i in range(Pk.shape[0] - 1):
                ax3d.plot(
                    Pk[i:i + 2, 0], Pk[i:i + 2, 1], Pk[i:i + 2, 2],
                    color=cols[i], linewidth=3.2, alpha=0.95,
                )
        else:
            ax3d.plot(Pk[:, 0], Pk[:, 1], Pk[:, 2], color=CH3_LIK_3D_POINT_COLOR, linewidth=3.0, alpha=0.9)
    plot_img = fig_to_image(fig, dpi=CH3_ANIM_DPI)
    plt.close(fig)
    nll = float(-loss_log_likelihood(ws, we, bb, study, exam, y))
    bvmin = float(ball_vmin if ball_vmin is not None else vmin)
    bvmax = float(ball_vmax if ball_vmax is not None else vmax)
    if measurements is not None:
        right = measurements
    else:
        right = ch4_we_are_here_blocks(
            ws, we, bb, nll,
            nll_vmin=bvmin, nll_vmax=bvmax,
            point_color=CH3_LIK_3D_POINT_COLOR,
            grad=grad, step_size=step_size,
        )
    if gd_bottom_blocks is not None:
        bottom = gd_bottom_blocks
        rails_key = None
    elif gd_formulas:
        bottom = ch4_formula_blocks_gd_story(bold_update_idx=gd_bold_update_idx)
        rails_key = None if gd_bold_update_idx is not None else (
            ch4_rails_cache_key(gd_formulas=True) if float(write_progress) >= 1.0 - 1e-9 else None
        )
    elif notation_condensed:
        bottom = ch4_cached_formula_blocks_3d_story()
        rails_key = ch4_rails_cache_key(gd_formulas=False) if float(write_progress) >= 1.0 - 1e-9 else None
    else:
        bottom = ch4_cached_formula_blocks_3d_story()
        rails_key = ch4_rails_cache_key(gd_formulas=False) if float(write_progress) >= 1.0 - 1e-9 else None
    if notation_condensed:
        return compose_tutorial(
            plot_img,
            right_blocks=right,
            bottom_blocks=bottom,
            corner_blocks=ch4_cached_notation_corner_blocks(),
            right_title=CH4_HERE_SECTION_TITLE,
            bottom_title=CH4_FORMULAS_SECTION_TITLE,
            corner_title=CH4_NOTATION_SECTION_TITLE,
            right_title_color=CH3_LIK_3D_POINT_COLOR,
            write_progress=write_progress,
            plot_alpha=plot_alpha,
            theme="classic_light",
            rails_cache_key=rails_key,
        )
    return compose_tutorial(
        plot_img,
        right_blocks=right,
        bottom_blocks=bottom,
        corner_blocks=ch4_cached_notation_corner_blocks(),
        right_title=CH4_HERE_SECTION_TITLE,
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        right_title_color=CH3_LIK_3D_POINT_COLOR,
        write_progress=write_progress,
        plot_alpha=plot_alpha,
        theme="classic_light",
        rails_cache_key=rails_key,
    )


def _ch3_lik_we_are_here_at(pack, ws, we, bb):
    from ch4_layout import ch4_we_are_here_blocks

    nll = float(-loss_log_likelihood(ws, we, bb, pack["study"], pack["exam"], pack["y"]))
    return ch4_we_are_here_blocks(
        ws, we, bb, nll,
        nll_vmin=pack["ball_vmin"], nll_vmax=pack["ball_vmax"],
        point_color=CH3_LIK_3D_POINT_COLOR,
    )


def _ch3_lik_append_path_frames(
    frames, pack, *,
    show_ball_vectors=False,
    cam_rot_deg=CH3_LIK_3D_CAM_PATH_ROT,
    gd_formulas=False,
):
    path = pack["path"]
    path_us = pack["path_us"]
    n_path = max(len(path_us) - 1, 1)
    cols = pack["path_colors"]
    for i, pu in enumerate(path_us):
        pu = float(pu)
        idx = min(len(path) - 1, max(0, int(round(pu * (len(path) - 1)))))
        ws_i, we_i, bb_i = path[idx]
        frames.append(
            ch3_frame_lik_weight3d_measurements(
                pack["study"], pack["exam"], pack["y"], ws_i, we_i, bb_i,
                W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
                vmin=pack["vmin"], vmax=pack["vmax"],
                path_xyz=path, path_colors=cols, path_u=pu,
                notation_condensed=True,
                measurements=_ch3_lik_we_are_here_at(pack, ws_i, we_i, bb_i),
                ax3d_bounds=pack["ax3d_bounds"],
                wide_bounds=pack["ax3d_bounds"],
                show_ball_vectors=show_ball_vectors,
                ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
                cam_azim_u=float(i) / float(n_path),
                cam_rot_deg=float(cam_rot_deg),
                gd_formulas=gd_formulas,
            )
        )


def ch3_build_frames_likelihood_3d_measurements_story():
    pack = _ch3_lik_3d_measurements_pack()
    frames = []
    _ch3_lik_append_path_frames(frames, pack, show_ball_vectors=False)
    if frames:
        last = frames[-1]
        for _ in range(max(10, CH3_SCRIPT_N_HOLD // 3)):
            frames.append(last.copy())
    return frames


def _ch3_lik_gd_path(study, exam, y, start, n_steps, eta):
    """Gradient-descent trajectory from ``start`` in (w_ST, w_EL, b)."""
    ws, we, bb = float(start[0]), float(start[1]), float(start[2])
    pts = [[ws, we, bb]]
    eta = float(eta)
    for _ in range(int(n_steps)):
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        ws -= eta * float(g1)
        we -= eta * float(g2)
        bb -= eta * float(gb)
        pts.append([ws, we, bb])
    return np.asarray(pts, dtype=float)


def _ch3_lik_3d_gd_pack():
    pack = dict(_ch3_lik_3d_measurements_pack(ball_colormap_limits=False))
    trail_path = _ch3_lik_gd_path(
        pack["study"], pack["exam"], pack["y"],
        CH3_LIK_3D_PATH_START, CH3_LIK_GD_N_ITERS, CH3_LIK_GD_STEP,
    )
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    path_nll = np.array([
        float(-loss_log_likelihood(float(r[0]), float(r[1]), float(r[2]), study, exam, y))
        for r in trail_path
    ], dtype=float)
    pack.update({
        "gd_start": CH3_LIK_3D_PATH_START,
        "gd_n_iters": CH3_LIK_GD_N_ITERS,
        "gd_eta": CH3_LIK_GD_STEP,
        "ball_vmin": float(np.min(path_nll)),
        "ball_vmax": float(np.max(path_nll)),
        "path": trail_path,
    })
    return pack


def _ch3_lik_gd_render_frame(pack, spec, *, cam_azim_u=0.0):
    from ch4_layout import ch4_we_are_here_blocks

    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws, we, bb = float(spec["ws"]), float(spec["we"]), float(spec["bb"])
    grad = spec["grad"]
    eta = float(spec["eta"])
    nll = float(-loss_log_likelihood(ws, we, bb, study, exam, y))
    right = ch4_we_are_here_blocks(
        ws, we, bb, nll,
        nll_vmin=pack["ball_vmin"], nll_vmax=pack["ball_vmax"],
        point_color=CH3_LIK_3D_POINT_COLOR,
        grad=grad, step_size=eta,
    )
    trail = np.asarray(spec["trail"], dtype=float)
    path_colors = None
    if trail.shape[0] >= 2:
        vir = mpl.colormaps.get_cmap("viridis")
        span = max(float(pack["ball_vmax"]) - float(pack["ball_vmin"]), 1e-9)
        path_colors = [
            vir(float((
                float(-loss_log_likelihood(float(r[0]), float(r[1]), float(r[2]), study, exam, y))
                - float(pack["ball_vmin"])
            ) / span))
            for r in trail
        ]
    return ch3_frame_lik_weight3d_measurements(
        study, exam, y, ws, we, bb,
        W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
        vmin=pack["vmin"], vmax=pack["vmax"],
        path_xyz=trail if trail.shape[0] >= 2 else None,
        path_colors=path_colors,
        path_u=1.0,
        notation_condensed=True,
        measurements=right,
        ax3d_bounds=pack["ax3d_bounds"],
        wide_bounds=pack["ax3d_bounds"],
        show_ball_vectors=False,
        ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
        cam_azim_u=float(cam_azim_u),
        grad=grad,
        step_size=eta,
        gd_formulas=True,
        gd_arrows_grad=grad,
        gd_arrows_visible=spec["arrows"],
        gd_bold_update_idx=spec["bold"],
    )


def _ch3_lik_ball_intro_frames(pack, *, gd_formulas=False):
    """Zoom → 360° spin → zoom out; shared by ch4_05 and ch4_06."""
    _BALL_FIELD_CACHE.clear()
    path = pack["path"]
    ws0, we0, bb0 = path[0]
    wide = pack["ax3d_bounds"]
    tight = _ch3_lik_ball_zoom_bounds(ws0, we0, bb0, wide)
    intro_ball = _ch3_lik_ball_vector_field_cached(
        pack["study"], pack["exam"], pack["y"], ws0, we0, bb0, wide,
    )
    frames = []

    def _intro_frame(bounds, azim, *, path_u=0.0):
        if gd_formulas:
            from ch4_layout import ch4_we_are_here_blocks
            g1, g2, gb = _ch3_nll_sum_grad_at_point(
                pack["study"], pack["exam"], pack["y"], ws0, we0, bb0,
            )
            nll0 = float(-loss_log_likelihood(ws0, we0, bb0, pack["study"], pack["exam"], pack["y"]))
            right = ch4_we_are_here_blocks(
                ws0, we0, bb0, nll0,
                nll_vmin=pack["ball_vmin"], nll_vmax=pack["ball_vmax"],
                point_color=CH3_LIK_3D_POINT_COLOR,
                grad=(g1, g2, gb), step_size=CH3_LIK_GD_STEP,
            )
        else:
            right = _ch3_lik_we_are_here_at(pack, ws0, we0, bb0)
        return ch3_frame_lik_weight3d_measurements(
            pack["study"], pack["exam"], pack["y"], ws0, we0, bb0,
            W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
            vmin=pack["vmin"], vmax=pack["vmax"],
            path_xyz=path, path_u=path_u,
            notation_condensed=True,
            measurements=right,
            ax3d_bounds=bounds,
            wide_bounds=wide,
            show_ball_vectors=True,
            ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
            ball_field=intro_ball,
            azim=float(azim),
            gd_formulas=gd_formulas,
        )

    for tv in np.linspace(0.0, 1.0, CH3_LIK_3D_N_INTRO_ZOOM, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_intro_frame(_ch3_lik_lerp_bounds(wide, tight, u), CH3_LIK_3D_CAM_AZIM0))

    spin_n = max(int(CH3_LIK_3D_N_INTRO_SPIN), 2)
    for tv in np.linspace(0.0, 1.0, spin_n, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_intro_frame(tight, CH3_LIK_3D_CAM_AZIM0 + 360.0 * float(u)))

    for tv in np.linspace(0.0, 1.0, CH3_LIK_3D_N_INTRO_ZOOM_OUT, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_intro_frame(_ch3_lik_lerp_bounds(tight, wide, u), CH3_LIK_3D_CAM_AZIM0))

    return frames


def _ch3_lik_append_gd_frames(
    frames, pack, *,
    cam_rot_deg=CH3_LIK_3D_CAM_PATH_ROT,
):
    specs = _ch3_lik_gd_frame_specs(pack)
    n = max(len(specs), 1)
    for i, spec in enumerate(specs):
        frames.append(_ch3_lik_gd_render_frame(
            pack, spec, cam_azim_u=float(i) / float(n),
        ))


def _ch3_lik_story_hold(frames):
    if frames:
        last = frames[-1]
        for _ in range(max(10, CH3_SCRIPT_N_HOLD // 3)):
            frames.append(last.copy())
    return frames


def _ch3_lik_pack_path_frame(pack, path_index=0, *, show_ball_vectors=False, gd_formulas=False, **frame_kw):
    """Render one path frame from a measurements/GD pack (no full story build)."""
    path = pack["path"]
    path_us = pack["path_us"]
    i = int(np.clip(path_index, 0, len(path_us) - 1))
    pu = float(path_us[i])
    idx = min(len(path) - 1, max(0, int(round(pu * (len(path) - 1)))))
    ws_i, we_i, bb_i = path[idx]
    n_path = max(len(path_us) - 1, 1)
    return ch3_frame_lik_weight3d_measurements(
        pack["study"], pack["exam"], pack["y"], ws_i, we_i, bb_i,
        W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
        vmin=pack["vmin"], vmax=pack["vmax"],
        path_xyz=path, path_colors=pack["path_colors"], path_u=pu,
        notation_condensed=True,
        measurements=_ch3_lik_we_are_here_at(pack, ws_i, we_i, bb_i),
        ax3d_bounds=pack["ax3d_bounds"],
        wide_bounds=pack["ax3d_bounds"],
        show_ball_vectors=show_ball_vectors,
        ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
        cam_azim_u=float(i) / float(n_path),
        gd_formulas=gd_formulas,
        **frame_kw,
    )


def ch4_preview_likelihood_notation_nll_last_frame():
    """Last ch4_03 frame for layout checks — does not build the full MP4 story."""
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_LIK_PLOT_START_RECT,
        CH4_NOTATION_SECTION_TITLE,
        ch4_cached_notation_corner_blocks,
        ch4_formula_blocks_nll_story,
        compose_tutorial,
    )

    state = _ch3_lik86_terminal_state()
    plot = ch3_frame_lik_w12_single_surface(
        state,
        log_u=1.0,
        nll_u=1.0,
        show_axis_labels=True,
        knob_labeled_blend=(1.0, 1.0, 1.0),
    )
    return compose_tutorial(
        plot,
        right_blocks=[],
        bottom_blocks=ch4_formula_blocks_nll_story(),
        corner_blocks=ch4_cached_notation_corner_blocks(),
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        layout_u=1.0,
        panel_u=1.0,
        write_progress=1.0,
        plot_start_rect=CH4_LIK_PLOT_START_RECT,
        theme="classic_light",
    )


def ch4_preview_likelihood_3d_measurements_frame(path_index=0):
    """Single ch4_04 frame for layout checks — does not build the full MP4 story."""
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=False)
    return _ch3_lik_pack_path_frame(pack, path_index, show_ball_vectors=False, gd_formulas=False)


def ch4_preview_likelihood_3d_ball_vectors_frame(*, intro=True, path_index=0):
    """Single ch4_05 frame — intro zoom start or one path index."""
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=True)
    if intro:
        path = pack["path"]
        ws0, we0, bb0 = path[0]
        wide = pack["ax3d_bounds"]
        intro_ball = _ch3_lik_ball_vector_field_cached(
            pack["study"], pack["exam"], pack["y"], ws0, we0, bb0, wide,
        )
        return ch3_frame_lik_weight3d_measurements(
            pack["study"], pack["exam"], pack["y"], ws0, we0, bb0,
            W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
            vmin=pack["vmin"], vmax=pack["vmax"],
            path_xyz=path, path_u=0.0,
            notation_condensed=True,
            measurements=_ch3_lik_we_are_here_at(pack, ws0, we0, bb0),
            ax3d_bounds=wide,
            wide_bounds=wide,
            show_ball_vectors=True,
            ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
            ball_field=intro_ball,
            azim=float(CH3_LIK_3D_CAM_AZIM0),
            gd_formulas=False,
        )
    return _ch3_lik_pack_path_frame(pack, path_index, show_ball_vectors=True, gd_formulas=False)


def ch4_preview_likelihood_3d_gd_frame(*, intro=True, gd_spec_index=0):
    """Single ch4_06 frame — first hold (all arrows) or one sequential-GD frame."""
    pack = _ch3_lik_3d_gd_pack()
    specs = _ch3_lik_gd_frame_specs(pack)
    idx = 0 if intro else int(np.clip(gd_spec_index, 0, len(specs) - 1))
    return _ch3_lik_gd_render_frame(pack, specs[idx], cam_azim_u=0.0)


def ch3_build_frames_likelihood_3d_ball_vectors_story():
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=True)
    frames = _ch3_lik_ball_intro_frames(pack, gd_formulas=False)
    _ch3_lik_append_path_frames(frames, pack, show_ball_vectors=True, gd_formulas=False)
    return _ch3_lik_story_hold(frames)


def ch3_build_frames_likelihood_3d_gd_story():
    pack = _ch3_lik_3d_gd_pack()
    frames = []
    _ch3_lik_append_gd_frames(frames, pack)
    return _ch3_lik_story_hold(frames)


def ch4_export_likelihood_notation_nll():
    frames = ch3_build_frames_likelihood_ch4_nll_story()
    fn = "ch4_03_likelihood_notation_nll.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_CH4_MS))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


def ch4_export_likelihood_3d_measurements():
    frames = ch3_build_frames_likelihood_3d_measurements_story()
    fn = "ch4_04_likelihood_3d_measurements.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_3D_MS))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


def ch4_export_likelihood_3d_ball_vectors():
    frames = ch3_build_frames_likelihood_3d_ball_vectors_story()
    fn = "ch4_05_likelihood_3d_ball_vectors.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_3D_MS_BALL))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


def ch4_export_likelihood_3d_gd():
    frames = ch3_build_frames_likelihood_3d_gd_story()
    fn = "ch4_06_likelihood_3d_gd.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_3D_MS_GD))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


In [ ]:
# ch4_00 — layout template (dark_rails for readable contrast on screen)
_scene = ch4_tutorial_scene()
_frame = ch4_render_tutorial_frame(_scene, write_progress=1.0, theme=CH4_TEMPLATE_THEME)
ch4_save_png(_frame, OUTPUT_DIR / "ch4_00_template_mistakes_triptych_layout.png")



In [ ]:
# ch4_01 — classic handwrite demo
ch4_export_handwrite_demo_mp4()

# ch4_02–06 — five color-theme variants (gradient backgrounds)
ch4_export_all_theme_demos()


### Likelihood story clips (ch4_02–06)

1. **ch4_02** — likelihood w₁₂ landscape (knob labels)
2. **ch4_03** — Ch4 template + Notation/Formulas + log → NLL
3. **ch4_04** — 3D weight space + We are here + NLL trajectory (90° camera pan)
4. **ch4_05** — ball of heatmapped gradient vectors + intro zoom/spin
5. **ch4_06** — sequential GD: colored axis arrows, per-parameter updates (10 steps)


In [ ]:
# ch4_02 — likelihood landscape (like ch3_83, but likelihood)
ch4_export_likelihood_w12_landscape()

# ch4_06 — gradient descent with animated weight updates
ch4_export_likelihood_3d_gd()


In [64]:
# ch4_03 — resize into Ch4 template, notation, log/NLL morph
ch4_export_likelihood_notation_nll()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3000, 1900) to (3008, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_03_likelihood_notation_nll.mp4


PosixPath('renders/ch4_03_likelihood_notation_nll.mp4')

In [65]:
# ch4_04 — 3D (w_ST, w_EL, b) measurements + colormap path
ch4_export_likelihood_3d_measurements()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3000, 1900) to (3008, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_04_likelihood_3d_measurements.mp4


PosixPath('renders/ch4_04_likelihood_3d_measurements.mp4')

In [ ]:
# ch4_05 — ball of NLL-colored gradient vectors + intro zoom/spin
ch4_export_likelihood_3d_ball_vectors()


In [ ]:
# ch4_06 — gradient descent on NLL with animated weight updates
ch4_export_likelihood_3d_gd()
